# SQL practice 3: aggregation and grouping

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql/sql-practice-03-aggregation-and-grouping.ipynb)

14 exercises on aggregates, HAVING, conditional aggregation, subtotals and pivots. Each exercise names the [reference entries](sql-reference-03-aggregation-and-grouping.ipynb) it practises, shows the exact expected output, and hides a reference solution you can reveal when you are done or stuck.

*Last reviewed: November 2026, against Apache Spark 4.0 (Spark SQL).*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

A local Spark session, the shared tables as temporary views: `customers`, `orders`, `order_items`, `products`, `events` and `employees` (customer 4 has no region, customers 7 and 8 never ordered, `SKU-4014` has no product row, one event is an exact retry, two pairs of employees tie on salary, Dana out-earns her manager and Jo has no department), and the helpers from [SQL reference 3](sql-reference-03-aggregation-and-grouping.ipynb#setup-and-sample-data). Exercises that need other data create it in their input cell.

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
# Shared sample tables as temporary views. Each statement is ordinary Spark SQL.
TABLES = {
    "customers": """CREATE OR REPLACE TEMP VIEW customers AS SELECT * FROM VALUES
  (1, 'Acme Corp',      'North', DATE'2024-01-03'),
  (2, 'Bright Retail',  'North', DATE'2024-02-11'),
  (3, 'Cyan Traders',   'South', DATE'2024-03-15'),
  (4, 'Delta Grocers',  NULL,    DATE'2024-05-02'),   -- region never arrived from the CRM
  (5, 'Echo Supplies',  'West',  DATE'2024-06-20'),
  (6, 'Foxtrot Foods',  'West',  DATE'2024-08-01'),
  (7, 'Gamma Sports',   'East',  DATE'2024-09-14'),   -- 7 and 8 have never ordered
  (8, 'Hotel Homeware', 'East',  DATE'2024-11-05')
AS customers(customer_id, name, region, signup_date)""",
    "orders": """CREATE OR REPLACE TEMP VIEW orders AS SELECT order_id, customer_id, status, order_ts, CAST(amount AS DECIMAL(12,2)) AS amount FROM VALUES
  (1,  1, 'paid',     TIMESTAMP'2026-01-05 09:15:00', 120.00),
  (2,  1, 'shipped',  TIMESTAMP'2026-01-05 10:05:00', 84.50),
  (3,  1, 'returned', TIMESTAMP'2026-01-05 11:40:00', 30.00),
  (4,  2, 'paid',     TIMESTAMP'2026-01-06 15:20:00', 260.00),
  (5,  2, 'placed',   TIMESTAMP'2026-01-07 08:05:00', 55.25),
  (6,  3, 'shipped',  TIMESTAMP'2026-01-07 17:45:00', 149.00),
  (7,  3, 'paid',     TIMESTAMP'2026-01-09 12:10:00', 310.00),
  (8,  4, 'returned', TIMESTAMP'2026-01-10 19:30:00', 89.90),
  (9,  4, 'shipped',  TIMESTAMP'2026-01-12 08:55:00', 1180.00),
  (10, 5, 'paid',     TIMESTAMP'2026-01-12 13:00:00', 74.00),
  (11, 6, 'placed',   TIMESTAMP'2026-01-13 10:25:00', 210.00),
  (12, 6, 'shipped',  TIMESTAMP'2026-01-14 00:30:00', 96.40)   -- just after midnight UTC
AS orders(order_id, customer_id, status, order_ts, amount)""",
    "order_items": """CREATE OR REPLACE TEMP VIEW order_items AS SELECT order_id, sku, qty, CAST(unit_price AS DECIMAL(8,2)) AS unit_price FROM VALUES
  (1, 'SKU-4001', 2, 12.50), (1, 'SKU-4007', 1, 95.00), (2, 'SKU-4002', 1, 34.50), (2, 'SKU-4009', 2, 25.00),
  (3, 'SKU-4002', 1, 30.00), (4, 'SKU-4003', 1, 149.00), (4, 'SKU-4004', 2, 55.50), (5, 'SKU-4005', 1, 55.25),
  (6, 'SKU-4003', 1, 149.00), (7, 'SKU-4006', 1, 189.00), (7, 'SKU-4008', 1, 121.00), (8, 'SKU-4010', 1, 89.90),
  (9, 'SKU-4011', 20, 59.00), (10, 'SKU-4012', 2, 37.00), (11, 'SKU-4013', 1, 210.00),
  (12, 'SKU-4014', 1, 41.40), (12, 'SKU-4005', 1, 55.00)
AS order_items(order_id, sku, qty, unit_price)""",
    "products": """CREATE OR REPLACE TEMP VIEW products AS SELECT sku, category, CAST(list_price AS DECIMAL(8,2)) AS list_price FROM VALUES
  ('SKU-4001', 'Kitchen', 12.50), ('SKU-4002', 'Kitchen', 34.50), ('SKU-4003', 'Garden', 149.00),
  ('SKU-4004', 'Garden', 55.50), ('SKU-4005', 'Pantry', 55.25), ('SKU-4006', 'Outdoor', 189.00),
  ('SKU-4007', 'Kitchen', 95.00), ('SKU-4008', 'Outdoor', 121.00), ('SKU-4009', 'Pantry', 25.00),
  ('SKU-4010', 'Garden', 89.90), ('SKU-4011', 'Pantry', 59.00), ('SKU-4012', 'Kitchen', 37.00),
  ('SKU-4013', 'Outdoor', 210.00)   -- SKU-4014 has no product row
AS products(sku, category, list_price)""",
    "events": """CREATE OR REPLACE TEMP VIEW events AS SELECT * FROM VALUES
  (1, 'view', TIMESTAMP'2026-01-06 09:00:00'), (1, 'add_to_cart', TIMESTAMP'2026-01-06 09:02:00'),
  (1, 'purchase', TIMESTAMP'2026-01-06 09:10:00'), (2, 'view', TIMESTAMP'2026-01-07 10:00:00'),
  (2, 'purchase', TIMESTAMP'2026-01-07 10:40:00'), (3, 'view', TIMESTAMP'2026-01-06 11:15:00'),
  (3, 'view', TIMESTAMP'2026-01-08 11:20:00'), (4, 'add_to_cart', TIMESTAMP'2026-01-09 20:05:00'),
  (4, 'purchase', TIMESTAMP'2026-01-10 08:30:00'), (5, 'view', TIMESTAMP'2026-01-11 14:45:00'),
  (5, 'view', TIMESTAMP'2026-01-12 09:00:00'), (5, 'purchase', TIMESTAMP'2026-01-12 12:30:00'),
  (5, 'view', TIMESTAMP'2026-01-12 09:00:00')   -- an SDK retry committed this event twice
AS events(user_id, event, ts)""",
    "employees": """CREATE OR REPLACE TEMP VIEW employees AS SELECT * FROM VALUES
  (1,  'Ada',    'Engineering', CAST(NULL AS INT), 185000, DATE'2018-03-01'),
  (2,  'Brett',  'Engineering', 1,    150000, DATE'2019-07-15'),
  (3,  'Chen',   'Engineering', 1,    150000, DATE'2020-01-06'),
  (4,  'Dana',   'Engineering', 2,    155000, DATE'2022-09-12'),   -- out-earns her manager
  (5,  'Eli',    'Sales',       1,    130000, DATE'2019-02-01'),
  (6,  'Farah',  'Sales',       5,    95000,  DATE'2021-05-17'),
  (7,  'Gus',    'Sales',       5,    95000,  DATE'2023-01-09'),
  (8,  'Hana',   'Finance',     1,    140000, DATE'2017-11-20'),
  (9,  'Ivan',   'Finance',     8,    88000,  DATE'2024-04-01'),
  (10, 'Jo',     NULL,          8,    70000,  DATE'2025-06-30')   -- department not assigned yet
AS employees(emp_id, name, dept, manager_id, salary, hire_date)""",
}
for statement in TABLES.values():
    spark.sql(statement)

import contextlib
import io
import logging
import re


def show(query, n=50, **args):
    """Run a query and print up to n rows without truncating values. Keyword arguments bind :name parameters."""
    spark.sql(query, args=args or None).show(n, truncate=False)


def plan(query, mode="formatted"):
    """The plan EXPLAIN prints for a query, as text."""
    return spark.sql(f"EXPLAIN {mode.upper()} {query}").first()[0]


@contextlib.contextmanager
def expected_error():
    """Run a block that is meant to fail and print the error class instead of a stack trace."""
    spark.sparkContext.setLogLevel("OFF")
    for logger in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
        logging.getLogger(logger).setLevel(logging.CRITICAL)
    try:
        yield
    except Exception as error:
        lines = [line for line in str(error).splitlines() if line.strip()]
        message = next((line for line in lines if re.search(r"\[[A-Z][A-Z0-9_.]+\]", line)), lines[0] if lines else "")
        print(type(error).__name__ + ":", message.strip().split(". ")[0][:120])
    finally:
        spark.sparkContext.setLogLevel("ERROR")

## Exercises

6 easy, 7 medium, 1 hard. Write your query in the empty cell under each exercise, run it with `show(q)` or `spark.sql(q).show()`, and compare it with the expected output (columns, values and row order).

| # | exercise | difficulty | practises |
| --- | --- | --- | --- |
| 1 | [Revenue by status](#exercise-1-revenue-by-status) | Easy | [`GROUP BY` and `HAVING`](sql-reference-03-aggregation-and-grouping.ipynb#group-by-and-having) |
| 2 | [Repeat customers above a spend](#exercise-2-repeat-customers-above-a-spend) | Easy | [`GROUP BY` and `HAVING`](sql-reference-03-aggregation-and-grouping.ipynb#group-by-and-having) |
| 3 | [Headcount and pay per department](#exercise-3-headcount-and-pay-per-department) | Easy | [`GROUP BY` and `HAVING`](sql-reference-03-aggregation-and-grouping.ipynb#group-by-and-having), [Core aggregates](sql-reference-03-aggregation-and-grouping.ipynb#core-aggregates) |
| 4 | [Paid versus returned per customer](#exercise-4-paid-versus-returned-per-customer) | Easy | [Conditional aggregation and `FILTER`](sql-reference-03-aggregation-and-grouping.ipynb#conditional-aggregation-and-filter) |
| 5 | [Largest order and its status](#exercise-5-largest-order-and-its-status) | Medium | [Picking values: `max_by`, `min_by`, `any_value`](sql-reference-03-aggregation-and-grouping.ipynb#picking-values-max_by-min_by-any_value) |
| 6 | [Order history as a string](#exercise-6-order-history-as-a-string) | Easy | [Collecting values: `collect_list`, `listagg`, `string_agg`](sql-reference-03-aggregation-and-grouping.ipynb#collecting-values-collect_list-listagg-string_agg) |
| 7 | [Median salary per department](#exercise-7-median-salary-per-department) | Medium | [Percentiles and distribution](sql-reference-03-aggregation-and-grouping.ipynb#percentiles-and-distribution) |
| 8 | [Subtotals with ROLLUP](#exercise-8-subtotals-with-rollup) | Medium | [`GROUPING SETS`, `ROLLUP` and `CUBE`](sql-reference-03-aggregation-and-grouping.ipynb#grouping-sets-rollup-and-cube) |
| 9 | [Two groupings in one pass](#exercise-9-two-groupings-in-one-pass) | Medium | [`GROUPING SETS`, `ROLLUP` and `CUBE`](sql-reference-03-aggregation-and-grouping.ipynb#grouping-sets-rollup-and-cube) |
| 10 | [Orders per status as columns](#exercise-10-orders-per-status-as-columns) | Medium | [`PIVOT` and `UNPIVOT`](sql-reference-03-aggregation-and-grouping.ipynb#pivot-and-unpivot) |
| 11 | [Quarterly columns to rows](#exercise-11-quarterly-columns-to-rows) | Medium | [`PIVOT` and `UNPIVOT`](sql-reference-03-aggregation-and-grouping.ipynb#pivot-and-unpivot) |
| 12 | [Average of averages](#exercise-12-average-of-averages) | Medium | [Aggregation pitfalls](sql-reference-03-aggregation-and-grouping.ipynb#aggregation-pitfalls) |
| 13 | [Customers who never returned anything](#exercise-13-customers-who-never-returned-anything) | Easy | [Core aggregates](sql-reference-03-aggregation-and-grouping.ipynb#core-aggregates) |
| 14 | [Engaged users per day](#exercise-14-engaged-users-per-day) | Hard | [`GROUP BY` and `HAVING`](sql-reference-03-aggregation-and-grouping.ipynb#group-by-and-having) |

## Exercise 1: Revenue by status

**Difficulty:** Easy · about 3 min · **Practises:** [`GROUP BY` and `HAVING`](sql-reference-03-aggregation-and-grouping.ipynb#group-by-and-having)

*The first query of most dashboards.*

Return `status`, `orders` and `revenue` (sum of amount) per status, highest revenue first.

**Expected output**

| status | orders | revenue |
| --- | --- | --- |
| shipped | 4 | 1509.90 |
| paid | 4 | 764.00 |
| placed | 2 | 265.25 |
| returned | 2 | 119.90 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT status, count(*) AS orders, sum(amount) AS revenue
    FROM orders
    GROUP BY status
    ORDER BY revenue DESC
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 2: Repeat customers above a spend

**Difficulty:** Easy · about 5 min · **Practises:** [`GROUP BY` and `HAVING`](sql-reference-03-aggregation-and-grouping.ipynb#group-by-and-having)

*WHERE filters rows, HAVING filters groups.*

Return `customer_id`, `orders` and `total` for customers with at least two non-returned orders totalling more than 200, ordered by `total` descending.

**Expected output**

| customer_id | orders | total |
| --- | --- | --- |
| 3 | 2 | 459.00 |
| 2 | 2 | 315.25 |
| 6 | 2 | 306.40 |
| 1 | 2 | 204.50 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT customer_id, count(*) AS orders, sum(amount) AS total
    FROM orders
    WHERE status <> 'returned'
    GROUP BY customer_id
    HAVING count(*) >= 2 AND sum(amount) > 200
    ORDER BY total DESC
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 3: Headcount and pay per department

**Difficulty:** Easy · about 5 min · **Practises:** [`GROUP BY` and `HAVING`](sql-reference-03-aggregation-and-grouping.ipynb#group-by-and-having), [Core aggregates](sql-reference-03-aggregation-and-grouping.ipynb#core-aggregates)

*NULL keys form one group, which needs a label.*

Return `dept` (`Unassigned` for no department), `headcount` and `avg_salary` rounded to 0 decimals, ordered by `headcount` descending then `dept`.

**Expected output**

| dept | headcount | avg_salary |
| --- | --- | --- |
| Engineering | 4 | 160000.0 |
| Sales | 3 | 106667.0 |
| Finance | 2 | 114000.0 |
| Unassigned | 1 | 70000.0 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT coalesce(dept, 'Unassigned') AS dept, count(*) AS headcount, round(avg(salary), 0) AS avg_salary
    FROM employees
    GROUP BY coalesce(dept, 'Unassigned')
    ORDER BY headcount DESC, dept
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 4: Paid versus returned per customer

**Difficulty:** Easy · about 5 min · **Practises:** [Conditional aggregation and `FILTER`](sql-reference-03-aggregation-and-grouping.ipynb#conditional-aggregation-and-filter)

*Several metrics in one pass instead of one query each.*

Per `customer_id` with orders, return `paid_total` (sum of paid amounts, 0 if none), `returned_total` (0 if none) and `orders`. Order by `customer_id`.

**Expected output**

| customer_id | paid_total | returned_total | orders |
| --- | --- | --- | --- |
| 1 | 120.00 | 30.00 | 3 |
| 2 | 260.00 | 0.00 | 2 |
| 3 | 310.00 | 0.00 | 2 |
| 4 | 0.00 | 89.90 | 2 |
| 5 | 74.00 | 0.00 | 1 |
| 6 | 0.00 | 0.00 | 2 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT customer_id,
           coalesce(sum(amount) FILTER (WHERE status = 'paid'), 0)     AS paid_total,
           coalesce(sum(amount) FILTER (WHERE status = 'returned'), 0) AS returned_total,
           count(*) AS orders
    FROM orders
    GROUP BY customer_id
    ORDER BY customer_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 5: Largest order and its status

**Difficulty:** Medium · about 5 min · **Practises:** [Picking values: `max_by`, `min_by`, `any_value`](sql-reference-03-aggregation-and-grouping.ipynb#picking-values-max_by-min_by-any_value)

*"Which row had the max?" without a window function.*

Per `customer_id`, return `largest` (max amount), `largest_order_id` and `largest_status` (from the order with the largest amount). Order by `customer_id`.

**Expected output**

| customer_id | largest | largest_order_id | largest_status |
| --- | --- | --- | --- |
| 1 | 120.00 | 1 | paid |
| 2 | 260.00 | 4 | paid |
| 3 | 310.00 | 7 | paid |
| 4 | 1180.00 | 9 | shipped |
| 5 | 74.00 | 10 | paid |
| 6 | 210.00 | 11 | placed |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT customer_id, max(amount) AS largest,
           max_by(order_id, amount) AS largest_order_id,
           max_by(status, amount)   AS largest_status
    FROM orders
    GROUP BY customer_id
    ORDER BY customer_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 6: Order history as a string

**Difficulty:** Easy · about 5 min · **Practises:** [Collecting values: `collect_list`, `listagg`, `string_agg`](sql-reference-03-aggregation-and-grouping.ipynb#collecting-values-collect_list-listagg-string_agg)

*One readable line per customer for support tools.*

Per `customer_id`, return `history`: the statuses joined with ` > ` in `order_ts` order. Order by `customer_id`.

**Expected output**

| customer_id | history |
| --- | --- |
| 1 | paid > shipped > returned |
| 2 | paid > placed |
| 3 | shipped > paid |
| 4 | returned > shipped |
| 5 | paid |
| 6 | placed > shipped |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT customer_id, listagg(status, ' > ') WITHIN GROUP (ORDER BY order_ts) AS history
    FROM orders
    GROUP BY customer_id
    ORDER BY customer_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 7: Median salary per department

**Difficulty:** Medium · about 5 min · **Practises:** [Percentiles and distribution](sql-reference-03-aggregation-and-grouping.ipynb#percentiles-and-distribution)

*Medians resist outliers that skew averages.*

Return `dept`, `median_salary` and `avg_salary` (rounded to 0) for employees with a department, ordered by `dept`.

**Expected output**

| dept | median_salary | avg_salary |
| --- | --- | --- |
| Engineering | 152500.0 | 160000.0 |
| Finance | 114000.0 | 114000.0 |
| Sales | 95000.0 | 106667.0 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT dept, median(salary) AS median_salary, round(avg(salary), 0) AS avg_salary
    FROM employees
    WHERE dept IS NOT NULL
    GROUP BY dept
    ORDER BY dept
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 8: Subtotals with ROLLUP

**Difficulty:** Medium · about 10 min · **Practises:** [`GROUPING SETS`, `ROLLUP` and `CUBE`](sql-reference-03-aggregation-and-grouping.ipynb#grouping-sets-rollup-and-cube)

*Finance reports want detail rows, subtotals and a grand total in one result.*

Return `customer_id` (as a string, `All` on the grand total row), `status` (`All` on subtotal and total rows) and `revenue`, for customers 1 and 2 only, from `ROLLUP (customer_id, status)`. Order detail rows before their subtotal, customers ascending, statuses alphabetically, grand total last.

**Expected output**

| customer_id | status | revenue |
| --- | --- | --- |
| 1 | paid | 120.00 |
| 1 | returned | 30.00 |
| 1 | shipped | 84.50 |
| 1 | All | 234.50 |
| 2 | paid | 260.00 |
| 2 | placed | 55.25 |
| 2 | All | 315.25 |
| All | All | 549.75 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT customer_label AS customer_id, status_label AS status, revenue
    FROM (
        SELECT CASE WHEN grouping(customer_id) = 1 THEN 'All' ELSE CAST(customer_id AS STRING) END AS customer_label,
               CASE WHEN grouping(status) = 1 THEN 'All' ELSE status END AS status_label,
               sum(amount) AS revenue,
               grouping(customer_id) AS is_total, customer_id AS customer_key,
               grouping(status) AS is_subtotal, status AS status_key
        FROM orders
        WHERE customer_id IN (1, 2)
        GROUP BY ROLLUP (customer_id, status)
    )
    ORDER BY is_total, customer_key, is_subtotal, status_key
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 9: Two groupings in one pass

**Difficulty:** Medium · about 10 min · **Practises:** [`GROUPING SETS`, `ROLLUP` and `CUBE`](sql-reference-03-aggregation-and-grouping.ipynb#grouping-sets-rollup-and-cube)

*GROUPING SETS replaces a UNION ALL of separate aggregations.*

Return `dimension` (`dept` or `hire_year`), `value` (as a string; `Unassigned` for no department) and `headcount`, counting employees by department and, separately, by hire year. Order by `dimension`, `value`.

**Expected output**

| dimension | value | headcount |
| --- | --- | --- |
| dept | Engineering | 4 |
| dept | Finance | 2 |
| dept | Sales | 3 |
| dept | Unassigned | 1 |
| hire_year | 2017 | 1 |
| hire_year | 2018 | 1 |
| hire_year | 2019 | 2 |
| hire_year | 2020 | 1 |
| hire_year | 2021 | 1 |
| hire_year | 2022 | 1 |
| hire_year | 2023 | 1 |
| hire_year | 2024 | 1 |
| hire_year | 2025 | 1 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT CASE WHEN grouping(dept) = 0 THEN 'dept' ELSE 'hire_year' END AS dimension,
           CASE WHEN grouping(dept) = 0 THEN coalesce(dept, 'Unassigned') ELSE CAST(year(hire_date) AS STRING) END AS value,
           count(*) AS headcount
    FROM employees
    GROUP BY GROUPING SETS ((dept), (year(hire_date)))
    ORDER BY dimension, value
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 10: Orders per status as columns

**Difficulty:** Medium · about 5 min · **Practises:** [`PIVOT` and `UNPIVOT`](sql-reference-03-aggregation-and-grouping.ipynb#pivot-and-unpivot)

*Wide summaries for spreadsheets.*

Return one row per `customer_id` with columns `placed`, `paid`, `shipped` and `returned` holding the number of orders in each status (0 when none). Order by `customer_id`.

**Expected output**

| customer_id | placed | paid | shipped | returned |
| --- | --- | --- | --- | --- |
| 1 | 0 | 1 | 1 | 1 |
| 2 | 1 | 1 | 0 | 0 |
| 3 | 0 | 1 | 1 | 0 |
| 4 | 0 | 0 | 1 | 1 |
| 5 | 0 | 1 | 0 | 0 |
| 6 | 1 | 0 | 1 | 0 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT customer_id,
           coalesce(placed, 0) AS placed, coalesce(paid, 0) AS paid,
           coalesce(shipped, 0) AS shipped, coalesce(returned, 0) AS returned
    FROM (SELECT customer_id, status FROM orders)
    PIVOT (count(*) FOR status IN ('placed', 'paid', 'shipped', 'returned'))
    ORDER BY customer_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 11: Quarterly columns to rows

**Difficulty:** Medium · about 5 min · **Practises:** [`PIVOT` and `UNPIVOT`](sql-reference-03-aggregation-and-grouping.ipynb#pivot-and-unpivot)

*Spreadsheet exports arrive wide; models want them long.*

Using `quarterly` below, return `category`, `quarter` and `units` with one row per non-NULL quarter value, ordered by `category`, `quarter`.

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW quarterly AS SELECT * FROM VALUES
  ('Garden', 40, 55, NULL, 20), ('Kitchen', 10, 12, 15, 18)
AS quarterly(category, q1, q2, q3, q4)
""")
show("SELECT * FROM quarterly")

**Expected output**

| category | quarter | units |
| --- | --- | --- |
| Garden | q1 | 40 |
| Garden | q2 | 55 |
| Garden | q4 | 20 |
| Kitchen | q1 | 10 |
| Kitchen | q2 | 12 |
| Kitchen | q3 | 15 |
| Kitchen | q4 | 18 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT category, quarter, units
    FROM quarterly
    UNPIVOT (units FOR quarter IN (q1, q2, q3, q4))
    ORDER BY category, quarter
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 12: Average of averages

**Difficulty:** Medium · about 5 min · **Practises:** [Aggregation pitfalls](sql-reference-03-aggregation-and-grouping.ipynb#aggregation-pitfalls)

*A classic interview trap: the two numbers are different, and only one is the average order value.*

Return one row with `avg_order_value` (the mean order amount) and `avg_customer_average` (the mean of each customer's average order amount), both rounded to 2.

**Expected output**

| avg_order_value | avg_customer_average |
| --- | --- |
| 221.59 | 221.24 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT round((SELECT avg(amount) FROM orders), 2) AS avg_order_value,
           round(avg(customer_avg), 2) AS avg_customer_average
    FROM (SELECT customer_id, avg(amount) AS customer_avg FROM orders GROUP BY customer_id)
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 13: Customers who never returned anything

**Difficulty:** Easy · about 5 min · **Practises:** [Core aggregates](sql-reference-03-aggregation-and-grouping.ipynb#core-aggregates)

*bool_and answers "for all rows in the group" questions.*

Return the `customer_id`s whose every order is `paid` or `shipped`, ordered.

**Expected output**

| customer_id |
| --- |
| 3 |
| 5 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT customer_id
    FROM orders
    GROUP BY customer_id
    HAVING bool_and(status IN ('paid', 'shipped'))
    ORDER BY customer_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 14: Engaged users per day

**Difficulty:** Hard · about 15 min · **Practises:** [`GROUP BY` and `HAVING`](sql-reference-03-aggregation-and-grouping.ipynb#group-by-and-having)

*Two levels of aggregation: per user-day, then per day.*

Count, per calendar day of `events.ts`, the users who performed at least two **different** event types that day. Return `day` and `engaged_users` for days with at least one such user, ordered by `day`.

**Expected output**

| day | engaged_users |
| --- | --- |
| 2026-01-06 | 1 |
| 2026-01-07 | 1 |
| 2026-01-12 | 1 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT day, count(*) AS engaged_users
    FROM (
        SELECT user_id, to_date(ts) AS day
        FROM events
        GROUP BY user_id, to_date(ts)
        HAVING count(DISTINCT event) >= 2
    )
    GROUP BY day
    ORDER BY day
"""
result = spark.sql(q)
result.show(truncate=False)

**Next:** look anything up in [SQL reference 3: aggregation and grouping](sql-reference-03-aggregation-and-grouping.ipynb), or try mixed interview problems in [SQL interview drills](sql-13-interview-drills.ipynb).